<a href="https://colab.research.google.com/github/DanielHevdeli/hafifot-tiug/blob/main/LLM_as_annotator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
token = "personal-access-token"

In [ ]:
!git clone https://{token}@github.com/DanielHevdeli/hafifot-tiug.git

In [ ]:
pip install dspy

In [4]:
import pandas as pd
import os
from typing import Literal, List
import dspy
import requests
import json

In [5]:
posts_df = pd.read_csv('./hafifot-tiug/data/split_data/present.csv')

In [6]:
posts_df.head(2)

,question_id,length,date,text
0,114678,903,2016-04-10 20:05:00,"שלום , אני מאוד מקווה שתוכלו לעזור לי אני לא י..."
1,116349,888,2016-04-24 19:56:00,היי כולם \nיש לי בעיה הקשורה לתספורת שלי. \nאמ...


In [7]:
posts_df.iloc[3]['text']

'בערך מגיל 17 אני סובלת מחרדה, כאשר חלה החרפה קיצונית בגיל 18 עקב לימודים ומצב חברתי גרוע והשפלות יומיומיות  . בהתחלה לא הבנתי בכלל שמדובר בחרדה כי הסימנים היו פיזיולוגים והחששות תמיד היו. התחלתי "להתעלף"  , הרגשתי שאני לא מצליחה לנשום וגם כשאני כן מצליחה אז האוויר לא נכנס לי לריאות , היו לי סחרחורות שהייתי בטוחה שאני עומדת למות ולהתמוטט. נשלחתי למיון וכאשר לא מצאו לי כלום הניחו שזה חרדה.ככה העברתי את השנה האחרונה שלי בבית ספר, כיתה יב. כלומר, במיטה ובדיכאון ללא חברים ולומדת לבגרויות. \nכמה חודשים אחר כך עליתי על מדים והאמת שמצבי השתפר פלאים. אומנם החרדה לא נעלמה , ותמיד היו ניצוצות של חרדה אבל הצלחתי לעבור אותן. \nכהשתחררתי היה לי מין תהום ענקית , הרגשתי בחופש ורציתי כל היום לישון,מה שבצבא לא התאפשר לי וגם כן לא לפני . יצאתי מדי פעם עם חברות אבל גם זה אחכ הפסיק. אחרי כמה חודשים כאלה התחלתי את הפסיכומטרי ולצערי בגלל תחושות של חרדה והתמוטטות בשיעור דחיתי את המועד. אחרי הדחייה נכנסתי לדכאון וחרדה קשים , לא רציתי לצאת מהמיטה.אם פעם התקפים באו והלכו אז המצב שלי בחודשיים האחרונים היו 24/7. 

In [8]:
posts_df.iloc[2]['text']

'הגעתי לאתר הזה במקרה אחרי שקראתי טיפה מפוסטים של אנונימים פה—אני מקווה שגם אוכל למצוא פה מענה או כיוון לפתרון לבעיה שלי.  \nאני לא יודעת כל כך מאיפה להתחיל או מאיפה התחיל הסיפור אבל אני סובלת בערך כל חיי מחרדה ודיכאון. \nהייתי מודעת לעניין אבל מעולם לא חוויתי מצב קיצוני של חרדה משתקת לחלוטין. היו לי התקפים מזעזעים שבגללם הלכתי למיון.  \nאבל בקיצור, בתקופה האחרונה של כמעט שנה אני חוויתי אגרופוביה רצינית עם התקפי דכאון ומחשבות מזעזעות. ברמה של סיעוד. לא התקלחתי, כל היום הייתי ישנה ומסוגרת מתחת לשמיכה. \nהפסיכאטר נתן לי ציפרלקס .  \nאני מרגישה שיפור עם הדיכאון יחסית, ועם ההתקפי חרדה אבל עד היום האגרובופיה לא נעלמה, קשה לי לתפקד. אני רוצה לצאת החוצה,להכיר אנשים, לעבוד, לקבל רישיון, ללמוד. לא להיות למה שהפכתי. \nמיותר לציין,שניתקתי קשר עם כל סבוביי כך שאין לי חיי חברה.  \nאת כל הכסף שאין לי אני מוציאה על מוניות בדרך לפסיכאטר ובחזרה מימנו כי קשה לי לעלות על אוטובוסים. \nהפסקתי עם השיעורי נהיגה בגלל ההתקפים בנהיגה .  \nבקיצור,אם המצב ימשיך אני לא יודעת איך אוכל לשרוד ככה,גם מבחינה כספית וגם 

Let's try to classify each post to either **suicidal-risk** or **non-suicidal-risk**. It may help the publishers of the website to offer first-help to writers of post categorized as suicidal even before other people answer them.

# Annotate Posts

In [9]:
CLASSES = ["non-suicidal-risk", "suicidal-risk"]

In [10]:
class SRClassification(dspy.Signature):
    text: str = dspy.InputField(desc="Hebrew post to classify.")
    label: Literal["suicidal-risk", "non-suicidal-risk"] = dspy.OutputField(
        desc="Classification result."
    )

In [11]:
class SRClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predict = dspy.Predict(SRClassification)

    def forward(self, text: str) -> str:
        result = self.predict(text=text)
        return result.label

# Local LM

In [12]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME,
  device_map="auto",
)

In [15]:
class TinyHebrewLM(dspy.LM):
    def __init__(self, model, tokenizer):
        super().__init__("TinyHebrewLM")
        self._hf_model = model
        self.tokenizer = tokenizer

    def __call__(self, **kwargs):
        messages = kwargs.pop("messages")
        prompt = ""
        for msg in messages:
            role = msg.get("role")
            content = msg.get("content", "")
            prompt += f"{role.upper()}:\n{content}\n"
        # Tokenize and generate
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self._hf_model.device)
        prompt_len = inputs["input_ids"].shape[1]
        print("Prompt len: ", prompt_len)
        print("Input IDs shape: ", inputs["input_ids"].shape)
        outputs = self._hf_model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0,
            do_sample=False
        )
        print("Outputs shape: ", outputs.shape)
        generated_ids = outputs[0, prompt_len:]
        print("Generated IDs shape: ", generated_ids.shape)
        generated_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        print("Generated text: ", generated_text)
        print("--------------------- END Generated Text ---------------------")
        return clean_output(generated_text)

In [16]:
hebrew_lm = TinyHebrewLM(model, tokenizer)
dspy.configure(lm=hebrew_lm)

## For Direct Classification

In [17]:
def clean_output(model_output: str):
    print("clean_output input: ", model_output)
    import json, re
    # remove ```json fences
    cleaned = re.sub(r"```json\s*|\s*```", "", model_output, flags=re.IGNORECASE).strip()
    return json.loads(cleaned)

In [18]:
import random

def get_class(text: str):
    if CLASSES[0] in text:
        return CLASSES[0], 0
    elif CLASSES[1] in text:
        return CLASSES[1], 0
    else:
        return random.choice(CLASSES), 1

In [19]:
def classify(tokenizer, hf_model, prompt):
    # Tokenize, Generate, Clean
    inputs = tokenizer(prompt, return_tensors="pt").to(hf_model.device)
    prompt_len = inputs["input_ids"].shape[1]
    outputs = hf_model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0,
        do_sample=False
    )
    generated_ids = outputs[0, prompt_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return get_class(generated_text)

In [20]:
def direct_classify_post(user_text: str) -> str:
    system_instruction = f"SYSTEM: You are an expert classifier. Your task is to classify the given text into {CLASSES[0]} or {CLASSES[1]}. Output ONLY a JSON object with a single key 'label' and the classification result as its value. Do NOT include any other text, markdown, or explanations.\n"
    formatted_prompt = f"{system_instruction}USER:\n{user_text}\nASSISTANT:\n"
    return classify(tokenizer=tokenizer, hf_model=model, prompt=formatted_prompt)

## Using DSpy

In [21]:
# sr_classifier = SRClassifier()
# user_prompt = posts_df.iloc[3]['text']
# label = sr_classifier(user_prompt)
# print(label)

In [22]:
# sr_classifier = SRClassifier()
# posts_labels = []
# i = 0
# for index, row in posts_df.iterrows():
#     if i > 0: break
#     question_id = row['question_id']
#     text = row['text']

#     label = sr_classifier(text)
#     posts_labels.append({'question_id': question_id, 'label': label})
#     i += 1

# print(f"Classification of {len(posts_labels)} posts completed.")

## Direct

In [ ]:
from tqdm.notebook import tqdm

posts_labels = []

for index, row in tqdm(posts_df.iterrows(), total=len(posts_df), desc="Classifying posts"):
    question_id = row['question_id']
    text = row['text']
    label, unknown = direct_classify_post(text)
    posts_labels.append({'question_id': question_id, 'label': label, "unknown": unknown})

print(f"Classification of {len(posts_labels)} posts completed.")

In [24]:
post_labels_df = pd.DataFrame(posts_labels)
display(post_labels_df.tail(10))

,question_id,label,unknown
3990,106463,non-suicidal-risk,0
3991,125396,non-suicidal-risk,0
3992,102887,non-suicidal-risk,0
3993,104960,non-suicidal-risk,0
3994,115122,non-suicidal-risk,0
3995,112514,non-suicidal-risk,0
3996,106940,suicidal-risk,0
3997,106899,non-suicidal-risk,0
3998,106285,non-suicidal-risk,0
3999,123863,non-suicidal-risk,0


In [25]:
MODEL_SHORT_NAME = MODEL_NAME.split('/')[-1]

save_dir = './hafifot-tiug/data/labels/present'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

save_path = f'{save_dir}/{MODEL_SHORT_NAME}.csv'
post_labels_df.to_csv(save_path, index=False)
print(f"{MODEL_SHORT_NAME} labels saved to {save_path} successfully.")

Qwen2.5-0.5B-Instruct labels saved to ./hafifot-tiug/data/labels/present/Qwen2.5-0.5B-Instruct.csv successfully.


## Save to Github

In [ ]:
%cd ./hafifot-tiug

In [ ]:
!git status

In [28]:
!git add .

In [29]:
!git config --global user.name "Daniel Hevdeli"

In [30]:
!git config --global user.email "daniel.hevdeli@gmail.com"

In [ ]:
!git commit -m "classify present.csv"

In [ ]:
!git push